In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import requests

In [7]:
project_path = ".."

raw_path = os.path.join(project_path, "data", "raw")
cleaned_path = os.path.join(project_path, "data", "cleaned")
plots_path = os.path.join(project_path, "plots")

print("Project:", os.path.abspath(project_path))
print("Raw data:", os.path.abspath(raw_path))
print("Cleaned data:", os.path.abspath(cleaned_path))
print("Plots:", os.path.abspath(plots_path))

Project: /home/darshani/lightkurve-env/quannt
Raw data: /home/darshani/lightkurve-env/quannt/data/raw
Cleaned data: /home/darshani/lightkurve-env/quannt/data/cleaned
Plots: /home/darshani/lightkurve-env/quannt/plots


In [8]:
os.makedirs(raw_path, exist_ok=True)
os.makedirs(cleaned_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

print("All project folders are ready!")

All project folders are ready!


In [32]:
import socket
import urllib3.util.connection as urllib3_cn

urllib3_cn.allowed_gai_family = lambda: socket.AF_INET
print("IPv4 forced")

IPv4 forced


In [33]:
import requests

url = "https://data-api.binance.vision/api/v3/klines"
test = requests.get(url, params={"symbol": "BTCUSDT", "interval": "1h", "limit": 5}, timeout=30)
print(test.status_code)
print(len(test.json()))

200
5


In [ ]:
url = "https://data-api.binance.vision/api/v3/klines"
# if you get an error, use this instead:
# url = "https://data-api.binance.vision/api/v3/klines"

end_time = int(time.time() * 1000)
start_time = end_time - 365 * 24 * 60 * 60 * 1000

all_data = []

while start_time < end_time:
    params = {
        "symbol": "BTCUSDT",
        "interval": "1h",
        "startTime": start_time,
        "limit": 1000
    }
    response = requests.get(url, params=params)
    batch = response.json()

    if len(batch) == 0:
        break

    all_data = all_data + batch
    start_time = batch[-1][0] + 1
    time.sleep(0.2)

print("Rows downloaded:", len(all_data))

Rows downloaded: 8760


In [35]:
import json
import os

raw_file_path = "../data/raw/bitcoin_hourly.json"

with open(raw_file_path, "w") as file:
    json.dump(all_data, file)

print("Raw data saved!")
print("File location:", os.path.abspath(raw_file_path))

Raw data saved!
File location: /home/darshani/lightkurve-env/quannt/data/raw/bitcoin_hourly.json


In [36]:
import json
import pandas as pd

with open("../data/raw/bitcoin_hourly.json", "r") as file:
    raw = json.load(file)

data = pd.DataFrame(raw)
data = data.iloc[:, :6]
data.columns = ["timestamp", "open", "high", "low", "price_usd", "volume"]

data["timestamp"] = pd.to_datetime(data["timestamp"], unit="ms")
data["open"] = data["open"].astype(float)
data["high"] = data["high"].astype(float)
data["low"] = data["low"].astype(float)
data["price_usd"] = data["price_usd"].astype(float)
data["volume"] = data["volume"].astype(float)

print("Shape:", data.shape)
print("Gap between rows:", data["timestamp"].diff().median())
print("From", data["timestamp"].iloc[0], "to", data["timestamp"].iloc[-1])
data.head()

Shape: (8760, 6)
Gap between rows: 0 days 01:00:00
From 2025-10-23 21:00:00 to 2026-10-23 20:00:00


,timestamp,open,high,low,price_usd,volume
0,2025-10-23 21:00:00,109524.81,109727.22,109380.12,109532.97,424.08473
1,2025-10-23 22:00:00,109532.96,110158.37,109517.37,109943.94,443.21336
2,2025-10-23 23:00:00,109943.95,110099.17,109839.00,110078.18,259.37538
3,2025-10-24 00:00:00,110078.19,110709.22,109983.93,110500.28,531.07612
4,2025-10-24 01:00:00,110500.29,110665.53,110141.38,110585.08,371.99990


In [37]:
data["timestamp"] = pd.to_datetime(data["timestamp"], unit="ms")

data["open"] = data["open"].astype(float)
data["high"] = data["high"].astype(float)
data["low"] = data["low"].astype(float)
data["price_usd"] = data["price_usd"].astype(float)
data["volume"] = data["volume"].astype(float)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  8760 non-null   datetime64[ns]
 1   open       8760 non-null   float64       
 2   high       8760 non-null   float64       
 3   low        8760 non-null   float64       
 4   price_usd  8760 non-null   float64       
 5   volume     8760 non-null   float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 410.8 KB


In [38]:
rows_before = len(data)

data = data.sort_values("timestamp")
data = data.drop_duplicates(subset="timestamp")
data = data.reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", len(data))
print("Duplicates removed:", rows_before - len(data))

Rows before: 8760
Rows after: 8760
Duplicates removed: 0


In [39]:
print("Missing values in each column:")
print(data.isnull().sum())

data = data.dropna().reset_index(drop=True)
print("\nRows after removing missing:", len(data))

Missing values in each column:
timestamp    0
open         0
high         0
low          0
price_usd    0
volume       0
dtype: int64

Rows after removing missing: 8760


In [40]:
# prices should never be zero or negative
bad_price = data[data["price_usd"] <= 0]

# high should never be lower than low
bad_high_low = data[data["high"] < data["low"]]

# volume should never be negative
bad_volume = data[data["volume"] < 0]

print("Bad prices:", len(bad_price))
print("Bad high/low rows:", len(bad_high_low))
print("Bad volume rows:", len(bad_volume))

# keep only the good rows
data = data[data["price_usd"] > 0]
data = data[data["high"] >= data["low"]]
data = data[data["volume"] >= 0]
data = data.reset_index(drop=True)

print("Rows left:", len(data))

Bad prices: 0
Bad high/low rows: 0
Bad volume rows: 0
Rows left: 8760


In [41]:
time_gap = data["timestamp"].diff()

print("Most common gap:", time_gap.mode()[0])
print("Biggest gap:", time_gap.max())

missing_hours = time_gap[time_gap > pd.Timedelta(hours=1)]
print("Number of gaps bigger than 1 hour:", len(missing_hours))

if len(missing_hours) > 0:
    print("\nWhere the gaps are:")
    print(data.loc[missing_hours.index, "timestamp"])

Most common gap: 0 days 01:00:00
Biggest gap: 0 days 01:00:00
Number of gaps bigger than 1 hour: 0


In [42]:
print("Final shape:", data.shape)
print("From", data["timestamp"].iloc[0], "to", data["timestamp"].iloc[-1])
print()
print(data.describe().round(2))

Final shape: (8760, 6)
From 2025-10-23 21:00:00 to 2026-10-23 20:00:00

                 timestamp       open       high        low  price_usd  \
count                 8760    8760.00    8760.00    8760.00    8760.00   
mean   2026-04-24 08:30:00   77906.66   78141.70   77659.17   77903.44   
min    2025-10-23 21:00:00   58291.70   58618.65   57800.19   58290.17   
25%    2026-01-23 02:45:00   66935.66   67183.53   66717.26   66935.64   
50%    2026-04-24 08:30:00   77335.29   77504.35   77159.06   77335.29   
75%    2026-07-24 14:15:00   86826.45   87168.22   86455.36   86815.92   
max    2026-10-23 20:00:00  116053.50  116400.00  115489.53  116053.49   
std                    NaN   12272.93   12297.63   12237.47   12268.34   

         volume  
count   8760.00  
mean     770.83  
min       46.43  
25%      339.65  
50%      541.97  
75%      911.36  
max    15965.68  
std      792.67  


In [44]:
os.makedirs("../data/cleaned", exist_ok=True)

cleaned_file_path = "../data/cleaned/bitcoin_cleaned.parquet"
data.to_parquet(cleaned_file_path, index=False)

print("Cleaned data saved!")
print("File location:", os.path.abspath(cleaned_file_path))

Cleaned data saved!
File location: /home/darshani/lightkurve-env/quannt/data/cleaned/bitcoin_cleaned.parquet


In [45]:
check = pd.read_parquet("../data/cleaned/bitcoin_cleaned.parquet")

print("Loaded shape:", check.shape)
print(check.dtypes)
check.head()

Loaded shape: (8760, 6)
timestamp    datetime64[ns]
open                float64
high                float64
low                 float64
price_usd           float64
volume              float64
dtype: object


,timestamp,open,high,low,price_usd,volume
0,2025-10-23 21:00:00,109524.81,109727.22,109380.12,109532.97,424.08473
1,2025-10-23 22:00:00,109532.96,110158.37,109517.37,109943.94,443.21336
2,2025-10-23 23:00:00,109943.95,110099.17,109839.00,110078.18,259.37538
3,2025-10-24 00:00:00,110078.19,110709.22,109983.93,110500.28,531.07612
4,2025-10-24 01:00:00,110500.29,110665.53,110141.38,110585.08,371.99990
